# AI Video Gen — LTX-Video on Kaggle

Rendering on Kaggle's free GPU quota (**30 hours/week**) instead of a per-clip hosted API.

Set your prompt in the next cell and Run All. Both stages run here: the `ltx` backend
gets the GPU it expects, and stage one enriches your prompt in place if you have given
the notebook an `HF_TOKEN` secret — otherwise the setup cell says what it fell back to
and why.

## Before you run anything

Three settings, none of them on by default, and none of which fails in a way that says so:

1. **Phone verification**, once, in Kaggle Settings. It gates **both** the GPU and
   Internet. Without it the session is quietly given a CPU worker with no network, no
   matter what the accelerator is set to — so if the accelerator says GPU and there is
   still no `nvidia-smi`, this is why.
2. **Accelerator → GPU T4 x2** (or P100). Otherwise every cell runs on CPU and a render
   takes hours instead of minutes.
3. **Internet → On.** Required to clone the repo and pull weights.

Optional, in **Add-ons → Secrets**:

- **`HF_TOKEN`** — a free Hugging Face token. This is what lets stage one enrich a prompt
  of your own, rather than only the ten prompts in the benchmark set.
- **`GITHUB_TOKEN`** — only if you are cloning a private fork.

Quota is billed by *session wall-clock*, not GPU work, so **stop the session when you
finish**. An idle notebook burns the same 30 hours as a busy one.


## 1. Your shot

Everything you would normally change lives in the next cell. Set it, then Run All — no
other cell needs editing.


In [ ]:
# ─── Set your shot here. Nothing below this cell needs editing. ──────────────

PROMPT = "a red fox hunting in a snowstorm"

STYLE = "nature_doc"      # cinematic, nature_doc, documentary, noir,
                          # anime, claymation, cyberpunk, retro_8mm
SEED = 1001               # same prompt + same seed = the same clip. Change it to reroll,
                          # keep it fixed to compare one deliberate edit against another.
WIDTH, HEIGHT = 512, 320  # prove the shot here, then raise it — 768x512 is the next step
DURATION = 3              # seconds. 3-5 is the sweet spot; longer drifts and costs more

# Stage one, which turns the line above into a full shot description.
#   "auto"        pick the best available, and say which — see the setup cell below
#   "hf"          Hugging Face; enriches anything, needs an HF_TOKEN secret
#   "fixture"     lookup table, and it only covers the ten benchmark prompts
#   "passthrough" no enrichment at all: your words go to the model as written
ENRICHER = "auto"


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit(
        "No GPU in this session. nvidia-smi is not installed when the accelerator "
        "is None.\n"
        "Two things to check, in this order:\n"
        "  1. Phone verification, once, in Kaggle Settings. It gates the GPU as well "
        "as Internet, and a session without it is silently given a CPU worker no "
        "matter what the accelerator is set to.\n"
        "  2. Sidebar (the < chevron, or the three-dot menu on mobile) -> "
        "Accelerator -> GPU T4 x2.\n"
        "Then re-run this cell."
    )

out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
    capture_output=True, text=True,
)
if out.returncode != 0:
    raise SystemExit(f"nvidia-smi failed ({out.returncode}): {out.stderr.strip()[:300]}")

name, mib = (f.strip() for f in out.stdout.strip().splitlines()[0].split(','))
VRAM_GB = int(mib) / 1024

# 13b-distilled does not fit 16 GB without offloading, which is slow enough to spend
# the quota you came here to save. 2b fits comfortably. This choice is carried into
# .env.local below - it is not advisory.
VARIANT = '13b-distilled' if VRAM_GB >= 22 else '2b-distilled'
print(f'{name} - {VRAM_GB:.0f} GB -> {VARIANT}')

## 2. Fetch the project and LTX-Video

Two separate clones on purpose. LTX-Video's inference dependencies are heavy and pinned;
the project shells out to its CLI rather than importing it, so upstream refactors cannot
break the pipeline and the two dependency sets never have to agree.


In [ ]:
import os, pathlib

WORK = pathlib.Path('/kaggle/working')
PROJECT = WORK / 'Video-Generation-'
LTX = WORK / 'LTX-Video'

# Public repo clones as-is. For a private one, add a Kaggle Secret named GITHUB_TOKEN
# (Add-ons -> Secrets) and this picks it up without putting the token in the notebook.
REPO = 'github.com/PravinderSamra/Video-Generation-'
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    origin = f'https://{_tok}@{REPO}.git'
except Exception:
    origin = f'https://{REPO}.git'

if not PROJECT.exists():
    !git clone --depth 1 {origin} {PROJECT}
if not LTX.exists():
    !git clone --depth 1 https://github.com/Lightricks/LTX-Video.git {LTX}

print('project:', PROJECT.exists(), '| ltx:', LTX.exists())


In [ ]:
# Heavy install: several minutes, and it pulls a matched torch. Expect pip to warn about
# resolver conflicts with Kaggle's preinstalled stack -- harmless, they are for packages
# this pipeline never imports.
!pip install -q -e '{LTX}[inference]' 2>&1 | tail -5
!pip install -q PyYAML 2>&1 | tail -2


## 3. Point the project at LTX

`.env.local` rather than `.env`: it is gitignored and overrides the environment, so a
container-specific path never ends up committed.


In [ ]:
PKG = PROJECT / 'AI Video Gen'   # note the spaces -- quote this path in shell commands

# .env.local rather than .env: gitignored, and it overrides the environment.
# LTX_VARIANT carries the choice from the GPU check; without it the backend defaults
# to 13b-distilled, which will not fit a 16 GB card.
settings = [f'LTX_REPO={LTX}', f'LTX_VARIANT={VARIANT}']

# HF_TOKEN, if you added one under Add-ons -> Secrets. It is what lets stage one enrich
# a prompt of your own here: Ollama needs a local server Kaggle does not have, and the
# fixture set covers only the ten benchmark prompts.
try:
    from kaggle_secrets import UserSecretsClient
    settings.append(f"HF_TOKEN={UserSecretsClient().get_secret('HF_TOKEN')}")
    have_hf = True
except Exception:
    have_hf = False

# Written, not printed. Notebook output is saved with the notebook and Kaggle notebooks
# are shareable, so echoing this file would publish the token in it.
(PKG / '.env.local').write_text('\n'.join(settings) + '\n')
print('LTX_REPO / LTX_VARIANT written |', 'HF_TOKEN found' if have_hf else 'no HF_TOKEN secret')

# Resolve "auto" once, out loud, so the run records which stage one actually ran.
if ENRICHER == 'auto':
    import yaml
    golden = yaml.safe_load(
        (PKG / 'prompts' / 'golden_enrichments.yaml').read_text(encoding='utf-8')
    )['enrichments']
    known = {k.strip().lower() for k in golden}
    if have_hf:
        ENRICHER, why = 'hf', 'HF_TOKEN is set, so any prompt can be enriched'
    elif PROMPT.strip().lower() in known:
        ENRICHER, why = 'fixture', 'no HF_TOKEN, but this prompt is in the benchmark set'
    else:
        ENRICHER, why = 'passthrough', (
            'no HF_TOKEN and this prompt is not in the benchmark set, so it will be sent '
            'to the model unenriched. Add an HF_TOKEN secret for a real stage one'
        )
    print(f'enricher: {ENRICHER} ({why})')
else:
    print(f'enricher: {ENRICHER} (set by hand)')


## 4. Confirm the backend is ready

`ltx` should read `ready`. If it says `LTX_REPO is not set`, the path above is wrong;
if it names a missing `inference.py`, the clone did not complete.


In [ ]:
%cd "{PKG}"
!python -m src.cli --check


## 5. Render

First run also downloads the weights, so it is much slower than steady state. Start small:
prove the path end to end at 512x320 before scaling resolution or duration.


In [ ]:
import subprocess, sys

# A list rather than a shell string: prompts carry apostrophes and quotes, and shell
# interpolation would mangle them or break the command outright.
render = subprocess.run([
    sys.executable, '-m', 'src.cli', PROMPT,
    '--enricher', ENRICHER,
    '--backend', 'ltx',
    '--style', STYLE,
    '--seed', str(SEED),
    '--width', str(WIDTH),
    '--height', str(HEIGHT),
    '--duration', str(DURATION),
])
if render.returncode != 0:
    raise SystemExit(f'render failed ({render.returncode}) — see the output above')


## 6. Look at it, then keep what matters

`/kaggle/working` survives the session and is downloadable from the Output panel; anything
outside it is gone when the session stops.


In [ ]:
import glob, base64
from IPython.display import HTML, display

clips = sorted(glob.glob('outputs/*.mp4'), key=os.path.getmtime)
if not clips:
    print('No clips. Check the render cell output above.')
else:
    latest = clips[-1]
    print(latest, f'{os.path.getsize(latest)/1e6:.1f} MB')
    b64 = base64.b64encode(open(latest, 'rb').read()).decode()
    display(HTML(f'<video controls width=512 src="data:video/mp4;base64,{b64}"></video>'))


In [ ]:
# Sidecars are small, diffable, and the only durable record of how a clip was made.
# Copy them somewhere the session cannot take with it.
!mkdir -p /kaggle/working/keep && cp outputs/*.json /kaggle/working/keep/ 2>/dev/null
!ls -la /kaggle/working/keep/ 2>/dev/null || echo 'no sidecars yet'


## Benchmark set

Ten fixed prompts at fixed seeds. This is the run worth spending quota on — it is what
makes a prompt-template change or a backend swap comparable rather than remembered.
Budget a few minutes per clip on a T4 at these settings, plus the one-off weight download.


In [ ]:
# Uncomment when a single render above has worked.
# !python -m src.benchmark --backend ltx --enricher fixture
# !python -m src.review outputs/benchmark
